# Search event by MJD

Search for an event at a given MJD in:
1. `HESE12.hdf5` — the reference HESE12 event selection
2. `EvtGen_merged.h5` — our v3 processing output

In [11]:
import tables
import pandas as pd
import numpy as np

In [12]:
TARGET_MJDS = [56211, 56666]

HESE12_PATH = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr/HESE12.hdf5"
EVTGEN_PATH = "/data/user/tvaneede/GlobalFit/reco_processing/data/hese/output/v3/merged/EvtGen_merged.h5"

## Load event headers

In [13]:
def load_header(path):
    with tables.open_file(path, "r") as f:
        return pd.DataFrame({
            "Run":   f.root.I3EventHeader.col("Run"),
            "Event": f.root.I3EventHeader.col("Event"),
            "MJD":   f.root.I3EventHeader.col("time_start_mjd"),
        })

df_hese12 = load_header(HESE12_PATH)
df_evtgen = load_header(EVTGEN_PATH)

print(f"HESE12.hdf5:     {len(df_hese12)} events, MJD [{df_hese12['MJD'].min():.1f}, {df_hese12['MJD'].max():.1f}]")
print(f"EvtGen_merged.h5: {len(df_evtgen)} events, MJD [{df_evtgen['MJD'].min():.1f}, {df_evtgen['MJD'].max():.1f}]")

HESE12.hdf5:     97 events, MJD [55351.5, 59792.2]
EvtGen_merged.h5: 188 events, MJD [55351.3, 60275.8]


## Search — floor(MJD) in TARGET_MJDS

In [14]:
for target in TARGET_MJDS:
    print(f"\n=== MJD {target} ===")
    for label, df in [("HESE12.hdf5", df_hese12), ("EvtGen_merged.h5", df_evtgen)]:
        match = df[df["MJD"].astype(int) == target].reset_index(drop=True)
        print(f"  {label}:")
        if len(match):
            print(f"    FOUND {len(match)} event(s)")
            print(match[["Run", "Event", "MJD"]].to_string(index=False))
        else:
            below = df[df["MJD"] < target].nlargest(2, "MJD")
            above = df[df["MJD"] > target].nsmallest(2, "MJD")
            nearest = pd.concat([below, above]).sort_values("MJD")[["Run", "Event", "MJD"]]
            print(f"    NOT FOUND — 2 below and 2 above:")
            print(nearest.to_string(index=False))


=== MJD 56211 ===
  HESE12.hdf5:
    NOT FOUND — 2 below and 2 above:
   Run    Event          MJD
119674  8449256 55979.255175
120421 44633110 56115.728357
120844  6079235 56221.342402
121240 72944671 56265.133868
  EvtGen_merged.h5:
    NOT FOUND — 2 below and 2 above:
   Run    Event          MJD
120421 44633110 56115.728369
120638  2924273 56176.391426
120844  6079235 56221.342414
120867  9198123 56228.605534

=== MJD 56666 ===
  HESE12.hdf5:
    NOT FOUND — 2 below and 2 above:
   Run    Event          MJD
122752 41309299 56501.164101
123217 66529770 56603.111685
123867 11659459 56671.877876
123986 77999595 56679.204468
  EvtGen_merged.h5:
    NOT FOUND — 2 below and 2 above:
   Run    Event          MJD
123217 66529770 56603.111697
123326 18647949 56628.568865
123867 11659459 56671.877888
123986 77999595 56679.204480


## All EvtGen events ordered by MJD

In [15]:
display(
    df_evtgen.sort_values("MJD").reset_index(drop=True)
)

,Run,Event,MJD
0,115994,2538090,55351.322214
1,115994,29874216,55351.465966
2,116528,52433389,55451.070748
3,116698,10198436,55477.393098
4,117371,31623515,55571.258536
...,...,...,...
183,138065,72143074,60116.080444
184,138069,72184188,60117.415932
185,138125,11333473,60132.707524
186,138134,50598681,60134.414603
